# ⚡ UNLET-ADAS: Free On-Demand GPU Demo (Colab)
### B.E. Major Project | SJBIT Bengaluru | CSE 2025-26
**GitHub:** https://github.com/DEEK-SHITH/UNLET-ADAS

Runs the real deployed app on a **free Colab GPU** and gives you a
temporary public link -- no Hugging Face account, no payment, no
always-on hosting cost. Perfect for showing "Sharp" mode's real
speed during a presentation/viva/demo, without the ongoing GPU
billing a permanent Hugging Face Spaces deployment would carry (see
the "Deploy to Hugging Face Spaces (GPU)" section of the main
README if you ever do want that instead).

**How it works:** Colab gives you a free T4 GPU for the session;
[Cloudflare's free Quick Tunnel](https://developers.cloudflare.com/cloudflare-one/connections/connect-networks/do-more-with-tunnels/trycloudflare/)
exposes the app running on it with a temporary `https://*.trycloudflare.com`
link, no account needed on either side.

**Trade-offs to know before you rely on this for a real demo:**
- The link only works while this Colab tab stays open and connected
  -- closing it, or Colab reclaiming the GPU, kills the demo. Run
  this a few minutes before you actually need to show it, and keep
  the tab open throughout.
- The link is temporary and changes every time you re-run Cell 3 --
  share the fresh one each time, don't bookmark it.
- Free Colab GPU availability isn't guaranteed on demand (same quota
  system as training) -- if none is free right when you need to
  demo, it'll just run on CPU instead (still works, just not the
  GPU speedup you're demoing).

| Cell | What it does |
|---|---|
| 1 | Setup -- clone the repo, install dependencies, check GPU |
| 2 | Download `cloudflared` (the free tunnel client) |
| 3 | Launch the app + tunnel, print your demo link |
| 4 | Stop the demo when you're done |

**Run cells top to bottom, then click the printed link.**


In [ ]:
# ============================================================
# CELL 1 -- Setup
# ============================================================

# Anti-disconnect -- run this first (a demo can sit idle between
# runs while you're presenting other slides)
from IPython.display import display, Javascript
display(Javascript('''
function ClickConnect(){
    var btns = document.querySelectorAll("colab-toolbar-button");
    for(var i=0;i<btns.length;i++){
        if(btns[i].id=="connect") btns[i].click();
    }
}
setInterval(ClickConnect, 55000)
'''))
print('Anti-disconnect active!')

import os
if not os.path.exists('/content/UNLET-ADAS'):
    !git clone https://github.com/DEEK-SHITH/UNLET-ADAS.git /content/UNLET-ADAS
    print('Repo cloned!')
else:
    !cd /content/UNLET-ADAS && git pull
    print('Repo updated!')

%cd /content/UNLET-ADAS
!pip install -q -r requirements.txt

import torch
if torch.cuda.is_available():
    print(f'\nGPU available: {torch.cuda.get_device_name(0)} -- Sharp mode will run at full speed.')
else:
    print('\nNo GPU assigned this session (Runtime > Change runtime type > T4 GPU, '
          'then re-run this cell) -- the demo will still work, just on CPU.')


In [ ]:
# ============================================================
# CELL 2 -- Download cloudflared (free tunnel, no account needed)
# ============================================================
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
!chmod +x /content/cloudflared
print('cloudflared ready.')


In [ ]:
# ============================================================
# CELL 3 -- Launch the app + tunnel, print your demo link
# Takes ~20-30s for the app to finish loading its models before the
# link is ready to click.
# ============================================================
import subprocess, sys, time, re

# Streamlit itself -- invoked via `sys.executable -m streamlit` rather
# than the bare 'streamlit' command: pip's console-script directory
# isn't always on PATH for subprocess.Popen in Colab even right after
# a successful `pip install streamlit`, which raises
# FileNotFoundError: [Errno 2] No such file or directory: 'streamlit'.
# -m sidesteps PATH entirely since it just needs the package
# importable in the current interpreter, which pip install guarantees.
streamlit_proc = subprocess.Popen(
    [sys.executable, '-m', 'streamlit', 'run', 'app/streamlit_app.py',
     '--server.port', '8501', '--server.headless', 'true'],
    stdout=open('/content/streamlit.log', 'w'), stderr=subprocess.STDOUT)

# Cloudflare quick tunnel pointed at it
tunnel_proc = subprocess.Popen(
    ['/content/cloudflared', 'tunnel', '--url', 'http://localhost:8501'],
    stdout=open('/content/cloudflared.log', 'w'), stderr=subprocess.STDOUT)

print('Starting app and tunnel', end='', flush=True)
demo_url = None
for _ in range(60):
    time.sleep(1)
    print('.', end='', flush=True)
    try:
        log = open('/content/cloudflared.log').read()
        m = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', log)
        if m:
            demo_url = m.group(0)
            break
    except FileNotFoundError:
        pass

print()
if demo_url:
    print('=' * 60)
    print(f'  DEMO LINK: {demo_url}')
    print('=' * 60)
    print('Open that link, wait for the app to finish loading models')
    print('(sidebar will show model status), then demo away.')
    print('Keep this Colab tab open for as long as you need the link live.')
else:
    print('Tunnel URL not found yet -- check /content/cloudflared.log '
          'and /content/streamlit.log for errors, or just re-run this cell.')


In [ ]:
# ============================================================
# CELL 4 -- Stop the demo (run when you're done)
# ============================================================
try:
    streamlit_proc.terminate()
    tunnel_proc.terminate()
    print('Demo stopped.')
except NameError:
    print('Nothing running (Cell 3 was not executed, or already stopped).')
